# 📊 Exploratory Data Analysis — Gym Exercises Dataset

Este notebook utiliza **KedroSession** para cargar datos del catálogo y genera
visualizaciones interactivas con Seaborn y Matplotlib.

**Requisito:** ejecuta primero `kedro run --pipeline=data_understanding` para
generar los datasets intermedios, o carga directamente los datos crudos.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis", font_scale=1.1)
%matplotlib inline

## 1. Carga de datos con KedroSession

In [ ]:
import os, sys

# Asegurar que el directorio raíz del proyecto esté en el path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, os.path.join(project_root, "src"))
os.chdir(project_root)

from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
from pathlib import Path

bootstrap_project(Path(project_root))

session = KedroSession.create()
context = session.load_context()
catalog = context.catalog

print("✅ Catálogo Kedro cargado correctamente")
print(f"Datasets disponibles: {catalog.list()}")

In [ ]:
# Cargar el dataset crudo
raw_df = catalog.load("raw_exercises_data")
print(f"Dimensiones: {raw_df.shape}")
raw_df.head(3)

In [ ]:
# Intentar cargar el dataset limpio (requiere haber ejecutado el pipeline)
try:
    clean_df = catalog.load("intermediate_exercises_clean")
    print(f"Dataset limpio cargado: {clean_df.shape}")
except Exception:
    print("⚠️  Dataset limpio no encontrado. Ejecutando flatten manualmente...")
    from gym_exercises.pipelines.data_understanding.nodes import flatten_exercise_metadata
    clean_df = flatten_exercise_metadata(raw_df)
    print(f"Dataset aplanado: {clean_df.shape}")

clean_df.head()

## 2. Resumen estadístico

In [ ]:
# Ejercicios únicos
n_unique = clean_df["id"].nunique()
print(f"Ejercicios únicos: {n_unique}")
print(f"\nNulos por columna:")
display(clean_df.isnull().sum().to_frame("nulls").query("nulls > 0"))

print(f"\nTipos de datos:")
display(clean_df.dtypes.to_frame("dtype"))

## 3. Distribución por Body Part (Parte del cuerpo)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

body_part_counts = (
    clean_df.drop_duplicates(subset=["id"])
    .groupby("body_part", as_index=False)
    .agg(count=("id", "nunique"))
    .sort_values("count", ascending=True)
)

sns.barplot(
    data=body_part_counts,
    x="count",
    y="body_part",
    hue="body_part",
    palette="viridis",
    legend=False,
    ax=ax,
)

for i, (_, row) in enumerate(body_part_counts.iterrows()):
    ax.text(row["count"] + 2, i, str(row["count"]), va="center", fontweight="bold")

ax.set_title("Ejercicios por Parte del Cuerpo", fontsize=14, fontweight="bold")
ax.set_xlabel("Cantidad de Ejercicios")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 4. Distribución por Equipment (Equipamiento)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

equip_counts = (
    clean_df.drop_duplicates(subset=["id"])
    .groupby("equipment", as_index=False)
    .agg(count=("id", "nunique"))
    .sort_values("count", ascending=True)
    .tail(15)
)

sns.barplot(
    data=equip_counts,
    x="count",
    y="equipment",
    hue="equipment",
    palette="magma",
    legend=False,
    ax=ax,
)

for i, (_, row) in enumerate(equip_counts.iterrows()):
    ax.text(row["count"] + 2, i, str(row["count"]), va="center", fontweight="bold")

ax.set_title("Top 15 — Ejercicios por Equipamiento", fontsize=14, fontweight="bold")
ax.set_xlabel("Cantidad de Ejercicios")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 5. Distribución por Target Muscle (Músculo objetivo)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

target_counts = (
    clean_df.drop_duplicates(subset=["id"])
    .groupby("target", as_index=False)
    .agg(count=("id", "nunique"))
    .sort_values("count", ascending=True)
    .tail(20)
)

sns.barplot(
    data=target_counts,
    x="count",
    y="target",
    hue="target",
    palette="crest",
    legend=False,
    ax=ax,
)

for i, (_, row) in enumerate(target_counts.iterrows()):
    ax.text(row["count"] + 1, i, str(row["count"]), va="center", fontweight="bold")

ax.set_title("Top 20 — Ejercicios por Músculo Objetivo", fontsize=14, fontweight="bold")
ax.set_xlabel("Cantidad de Ejercicios")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 6. Heatmap — Body Part × Equipment

In [ ]:
# Tomar solo los top 10 equipamientos para legibilidad
top_equip = (
    clean_df.drop_duplicates(subset=["id"])
    ["equipment"].value_counts().head(10).index.tolist()
)

heatmap_df = (
    clean_df.drop_duplicates(subset=["id"])
    .query("equipment in @top_equip")
    .groupby(["body_part", "equipment"])
    .agg(count=("id", "nunique"))
    .reset_index()
    .pivot(index="body_part", columns="equipment", values="count")
    .fillna(0)
    .astype(int)
)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    heatmap_df,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
)
ax.set_title(
    "Ejercicios por Parte del Cuerpo × Equipamiento (Top 10)",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("Equipamiento")
ax.set_ylabel("Parte del Cuerpo")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. Músculos secundarios más frecuentes

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

sec_counts = (
    clean_df.dropna(subset=["secondary_muscle"])
    .groupby("secondary_muscle", as_index=False)
    .agg(count=("id", "nunique"))
    .sort_values("count", ascending=True)
    .tail(15)
)

sns.barplot(
    data=sec_counts,
    x="count",
    y="secondary_muscle",
    hue="secondary_muscle",
    palette="flare",
    legend=False,
    ax=ax,
)

for i, (_, row) in enumerate(sec_counts.iterrows()):
    ax.text(row["count"] + 1, i, str(row["count"]), va="center", fontweight="bold")

ax.set_title(
    "Top 15 — Músculos Secundarios más Frecuentes",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("Ejercicios donde aparece")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

---

## 📝 Conclusiones

- El dataset contiene **1,324 ejercicios** únicos, sin duplicados.
- Los grupos más representados son **upper arms** y **upper legs**.
- El equipamiento más frecuente es **body weight** (~25% del total).
- Existen instrucciones en **10 idiomas** para cada ejercicio.
- La relación body_part × equipment muestra patrones esperados (e.g., barbells dominan en chest/back).